# EEG → CLIP embedding → Stable UnCLIP

Przed startem wybierz **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU**. Notebook używa istniejącego ZIP-a z epokami QC oraz `biai_unclip_assets.zip` z obrazami, manifestami i kodem.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Konfiguracja
DRIVE_DATA = '/content/drive/MyDrive/biai/data'
EPOCHS_ZIP = f'{DRIVE_DATA}/biai_eeg_qc_0_0p8.zip'
ASSETS_ZIP = f'{DRIVE_DATA}/biai_unclip_assets.zip'

PARTICIPANT = 'mole'
EPOCHS = 25
BATCH_SIZE = 128
STEPS = 20

RESULTS = '/content/drive/MyDrive/biai/results'
RETRIEVAL_DIR = f'{RESULTS}/unclip_{PARTICIPANT}_retrieval'
GENERATION_DIR = f'{RESULTS}/unclip_{PARTICIPANT}_generation'

# Najpierw ustaw 2, aby sprawdzić generowanie. Potem ustaw None dla wszystkich 44 obrazów.
MAX_IMAGES = 2

In [ ]:
# Automatyczny przebieg. Trening zapisuje checkpoint na Drive po każdej epoce i wznawia się po rozłączeniu.
from datetime import datetime
from pathlib import Path
import shutil
import subprocess
import zipfile

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU nie jest aktywne.')
print('GPU:', torch.cuda.get_device_name(0))

subprocess.run(['pip', 'install', '-q', 'diffusers==0.38.0', 'transformers', 'accelerate', 'safetensors', 'scikit-learn'], check=True)

root = Path('/content/biai_unclip')
if root.exists():
    shutil.rmtree(root)
root.mkdir()
for source in (Path(EPOCHS_ZIP), Path(ASSETS_ZIP)):
    if not source.is_file():
        raise FileNotFoundError(f'Brak pliku na Drive: {source}')
    with zipfile.ZipFile(source) as archive:
        archive.extractall(root)

manifest = root / f'reconstruction_manifests/participant_image_{PARTICIPANT}_no_abc'
embedding_dir = root / f'image_embeddings_unclip_participant_image_{PARTICIPANT}_no_abc'

def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write(f'\n[{datetime.now().isoformat(timespec="seconds")}] {" ".join(map(str, command))}\n')
        process = subprocess.Popen(command, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'Polecenie nie powiodło się: {command}')

if not embedding_dir.exists():
    run_logged([
        'python', '-u', 'scripts/extract_unclip_image_embeddings.py',
        '--manifest-dir', str(manifest), '--project-root', str(root),
        '--output-dir', str(embedding_dir), '--batch-size', '8',
    ], Path(RETRIEVAL_DIR) / 'training.log')

run_logged([
    'python', '-u', 'scripts/train_eeg_image_retrieval.py',
    '--manifest-dir', str(manifest), '--project-root', str(root),
    '--embedding-dir', str(embedding_dir), '--output-dir', RETRIEVAL_DIR,
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--checkpoint-every', '1', '--resume',
], Path(RETRIEVAL_DIR) / 'training.log')

generation_command = [
    'python', '-u', 'scripts/generate_unclip_from_eeg.py',
    '--retrieval-result-dir', RETRIEVAL_DIR, '--project-root', str(root),
    '--output-dir', GENERATION_DIR, '--num-inference-steps', str(STEPS), '--oracle',
]
if MAX_IMAGES is not None:
    generation_command += ['--max-images', str(MAX_IMAGES)]
generation_summary = Path(GENERATION_DIR) / 'unclip_generation_summary.json'
if generation_summary.is_file():
    print('Generacja jest już gotowa:', generation_summary)
elif Path(GENERATION_DIR).exists():
    raise RuntimeError('Katalog generacji jest niekompletny. Ustaw nową nazwę GENERATION_DIR albo usuń go ręcznie.')
else:
    run_logged(generation_command, Path(RESULTS) / f'unclip_{PARTICIPANT}_generation.log')
